### KNN с использованием LSH

In [44]:
import numpy as np
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

Для демонстрации сгенерируем датасет из нормального распределения, где  m  - число объектов,  n  - число признаков.  q  - вектор запроса (query), для которого мы ищем ближайших соседей.

In [45]:
m=1000
n=10

x = rng.normal(size=(m,n)) # датасет
q = rng.normal(size=n)

### Поиск с помощью стандартного KNN

In [46]:
def knn_search(query, data, k=5):
    dists = np.sqrt(np.sum((data - query)**2, axis=1)) # вычисляем расстояние
    # от объекта query до всех точек датасета
    inds = np.argsort(dists) # сортируем расстояния по возрастанию
    inds_k = inds[:k] # берем лучшие k точек с наименьшим расстоянием
    return data[inds_k], dists[inds_k]

Найдем ближайшие 5 соседей и замерим время

In [47]:
%%time

neighbors, dists = knn_search(q,x)
for i, (neighbors, dist) in enumerate(zip(neighbors, dists)):
    print(f'top {i+1}: dist = {dist}')

top 1: dist = 1.6747556181715648
top 2: dist = 2.3617661196642983
top 3: dist = 2.458244061979841
top 4: dist = 2.4652457753485866
top 5: dist = 2.509121028236157
CPU times: total: 0 ns
Wall time: 572 μs


### KNN с использованием LSH

Идея метода состоит в том, что:

Сначала при помощи LSH отбираем объекты, похожие на объект query.  
Затем при помощи KNN ищем ближайшие объекты к query только среди похожих, найденных на предыдущем шаге

Используем метод LSH с кодированием при помощи случайных проекций. Он состоит в следующем:

проводим несколько случайных гиперплоскостей  
для каждой плоскости: для каждого объекта ставим 1, если объект лежит выше плоскости, и 0 иначе  
тогда каждый объект кодируется вектором из 0 и 1, где длина вектора равна числу сгенерированных гиперплоскостей
Объекты похожи, если их кодировки совпадают.

generate_hyperplanes вычисляет количество случайных гиперплоскостей и генерирует их, основываясь на том, сколько в среднем мы хотим получать объектов в одной корзине после хеширования (bucket_size)

In [48]:
def generate_hyperplanes(data, bucket_size=16):
    m = data.shape[0]          # число объектов
    n = data.shape[1]          # числов признаков
    b = m // bucket_size       # количество корзин
    h = int(np.log2(b))        # количество гиперплоскостей
    H = rng.normal(size=(h,n)) # гиперплоскости, заданные своими параметрами
    return H

hamming_hash хеширует данные, основываясь на полученных гиперплоскостях, то есть кодирует объект вектором из 0 и 1.

Затем для удобства функция переводит вектор в число путем перевода из двоичной системы в десятичную.

Полученное число - это номер корзины, в которую попадает объект.

In [49]:
def hamming_hash(data, hyperplanes):
    b = len(hyperplanes)
    hash_key = (data @ hyperplanes.T) >= 0

    dec_vals = np.array([2**i for i in range(b)], dtype=int)
    hash_key = hash_key @ dec_vals

    return hash_key

Теперь мы умеем по каждому объекту определять номер корзины, в которую он попадает.

locality_sensitive_hash (LSH) создает словарь, где для каждой корзины содержатся элементы выборки, попадающие в эту корзину (хеш-таблицy).

In [50]:
def locality_sensitive_hash(data, hyperplanes):
    hash_vals = hamming_hash(data, hyperplanes)
    hash_table = {}
    for i, j in enumerate(hash_vals):
        if j not in hash_table:
            hash_table[j] = set()
        hash_table[j].add(i)
    return hash_table

Проверим работу функций

In [51]:
hyperplanes = generate_hyperplanes(x)

print(f'Всего плоскостей: {len(hyperplanes)}')
print(f'Нормали первой гиперплоскости: {hyperplanes[0]}')

Всего плоскостей: 5
Нормали первой гиперплоскости: [ 0.86580721 -0.19325001  0.4003944   0.74520698 -0.81386321 -0.70684838
  0.18972618  0.90408896 -1.16595693 -0.19794763]


Теперь посмотрим, как работает функция hamming_hash

In [52]:
hamming_hash(q, hyperplanes)
hash_table = locality_sensitive_hash(x, hyperplanes)
hash_table

{np.int64(13): {0,
  19,
  28,
  44,
  45,
  77,
  87,
  99,
  102,
  103,
  105,
  127,
  142,
  157,
  167,
  186,
  204,
  213,
  214,
  221,
  227,
  235,
  251,
  254,
  267,
  285,
  286,
  288,
  293,
  298,
  299,
  310,
  312,
  328,
  334,
  340,
  353,
  354,
  357,
  363,
  365,
  371,
  386,
  388,
  391,
  401,
  411,
  413,
  417,
  426,
  427,
  447,
  458,
  481,
  514,
  518,
  533,
  571,
  575,
  576,
  579,
  586,
  589,
  592,
  598,
  600,
  612,
  620,
  622,
  627,
  637,
  651,
  653,
  654,
  662,
  676,
  691,
  695,
  710,
  720,
  729,
  733,
  741,
  771,
  774,
  782,
  814,
  823,
  847,
  856,
  863,
  873,
  892,
  903,
  906,
  925,
  936,
  951,
  955,
  969,
  990},
 np.int64(30): {1,
  12,
  27,
  47,
  94,
  108,
  118,
  124,
  148,
  154,
  165,
  192,
  202,
  239,
  253,
  260,
  270,
  302,
  307,
  324,
  343,
  345,
  375,
  379,
  394,
  408,
  434,
  449,
  452,
  457,
  478,
  496,
  516,
  519,
  526,
  528,
  553,
  559,
  574,
  597,

Теперь реализуем алгоритм поиска KNN с использованием LSH:

Сначала при помощи LSH отбираем объекты, похожие на объект query  
Затем при помощи KNN ищем ближайшие объекты к query только среди похожих, найденных на предыдущем шаге

In [53]:
def approx_knn_search(query, data, k=5, bucket_size=16): # approx - приблизительно
    candidates = set()
    hyperplanes = generate_hyperplanes(data)
    hash_table = locality_sensitive_hash(data, hyperplanes) # формируем хеш-таблицу по датасету

    query_hash = hamming_hash(query, hyperplanes)
    if query_hash in hash_table:
        candidates = candidates.union(hash_table[query_hash])
    candidates = np.stack([data[i] for i in candidates], axis=0) # находим объекты, попадающие с query в одну корзину

    return knn_search(query, candidates, k=k)

In [54]:
%%time

neighbors, dists = approx_knn_search(q, x)
for i, (neighbor, dist) in enumerate(zip(neighbors, dists)):
    print(f"top {i + 1}: dist = {dist}")

top 1: dist = 2.7418702429131896
top 2: dist = 3.282560573449175
top 3: dist = 3.3601819938220316
top 4: dist = 3.429352542973233
top 5: dist = 3.4561855971966913
CPU times: total: 0 ns
Wall time: 884 μs


### Итог

В зависимости от случая, время выполнения варьируется. 

В чем разница? На маленьких данных точный алгоритм(KNN) работает лучше, но на больших данных приближенный алгоритм работает намного лучше, потому что потребляет в меньше памяти. К примеру, при 100000 объектах и 10000 признаках при точном поиске google collab крашнулся, потому что доступная память переполнилась, в то время как приближенный поиск отработал.